# Appendix 6. EDA 기초

## 1. Goal

이 notebook은 model input 후보 table을 처음 받았을 때 확인할 EDA 순서를 익히는 자료입니다. 분석 질문과 row grain을 먼저 고정하고 schema, target 분포, numeric·categorical 분포, 결측, 중복, 이상 범위, feature 간 관계와 train/validation 안정성을 차례로 확인합니다.

여기서 기초는 API를 짧게 훑는다는 뜻이 아닙니다. 기술 통계와 시각화가 답하는 질문, 필요한 가정과 결론의 한계를 이해하는 데 초점을 둡니다. EDA 결과는 feature 사용 여부를 자동으로 결정하지 않으며, 다음 Feature Engineering notebook에서 검토할 가설과 위험을 기록합니다.

## 2. Setup

pandas, NumPy, Matplotlib과 scikit-learn의 split helper를 사용합니다. 실제 PhysioNet data나 sealed test는 읽지 않으며, 관계와 결함을 의도적으로 넣은 합성 patient-level table을 고정 seed로 생성합니다. 합성 관계는 임상적 의미를 갖지 않습니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

BLUE = "#2F5D8C"
ORANGE = "#D97732"
GOLD = "#C6A15B"
INK = "#30343B"
GRID = "#D9DEE5"
RANDOM_SEED = 42


## 3. Steps

### 3-1. 질문, grain과 schema

#### 3-1-1. 분석 질문과 한 row의 의미

EDA를 시작하기 전에 population, 관측 sample, prediction 대상, observation window와 한 row의 grain을 문장으로 고정합니다. Population은 결론을 적용하려는 전체 대상이고 sample은 실제로 관측한 일부입니다. Grain은 통계량의 분모와 독립적인 관측 단위를 결정합니다. 이 예제의 grain은 `patient_id`당 한 row이고 target은 합성 binary outcome입니다. 같은 patient의 여러 측정 row를 독립 sample처럼 세면 표본 수와 불확실성을 과대평가할 수 있습니다. Feature는 prediction 시점까지 관측 가능한 값이어야 합니다.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
patient_count = 240
age = np.clip(rng.normal(63, 15, patient_count), 18, 92)
heart_rate_mean = rng.normal(86, 14, patient_count)
lactate_max = rng.lognormal(0.65, 0.5, patient_count)
urine_sum = rng.lognormal(7.0, 0.45, patient_count)
icu_type = rng.choice(["medical", "surgical", "cardiac"], patient_count)
noise_feature = rng.normal(0, 1, patient_count)

risk_signal = (
    0.035 * (age - 60)
    + 0.045 * (heart_rate_mean - 85)
    + 0.8 * (lactate_max - 2.0)
    - 0.0007 * (urine_sum - 1100)
    + 0.25 * (icu_type == "medical")
    + rng.normal(0, 0.8, patient_count)
)
target_probability = 1 / (1 + np.exp(-risk_signal))
target = rng.binomial(1, target_probability)

candidate_table = pd.DataFrame(
    {
        "patient_id": [f"P{number:04d}" for number in range(patient_count)],
        "age": age,
        "heart_rate__mean": heart_rate_mean,
        "heart_rate__mean_copy": heart_rate_mean * 1.02 + rng.normal(0, 0.4, patient_count),
        "lactate__max": lactate_max,
        "urine__sum": urine_sum,
        "noise_feature": noise_feature,
        "icu_type": icu_type,
        "future_outcome_score": target + rng.normal(0, 0.03, patient_count),
        "target": target,
    }
)
candidate_table.loc[rng.choice(patient_count, 28, replace=False), "lactate__max"] = np.nan
candidate_table.loc[5, "age"] = 999
raw_candidate_table = pd.concat(
    [candidate_table, candidate_table.iloc[[0]]], ignore_index=True
)
raw_candidate_table.head()


#### 3-1-2. shape, dtype, key와 범위 확인

`shape`, `dtypes`, `nunique`와 duplicate count를 함께 봅니다. dtype이 맞아도 값의 의미가 맞는 것은 아닙니다. age처럼 허용 범위가 있는 값은 contract와 비교하고, sentinel이나 불가능한 값은 결측 처리 정책에 따라 정리합니다.

In [ ]:
duplicate_patient_count = int(
    raw_candidate_table.duplicated("patient_id", keep=False).sum()
)
invalid_age_count = int(raw_candidate_table["age"].gt(100).sum())
schema_summary = pd.DataFrame(
    {
        "dtype": raw_candidate_table.dtypes.astype(str),
        "non_null": raw_candidate_table.notna().sum(),
        "unique": raw_candidate_table.nunique(dropna=True),
    }
)
print(
    {
        "shape": raw_candidate_table.shape,
        "duplicate_patient_rows": duplicate_patient_count,
        "invalid_age_rows": invalid_age_count,
    }
)
schema_summary


In [ ]:
eda_table = (
    raw_candidate_table.drop_duplicates("patient_id", keep="first")
    .assign(
        age=lambda frame: frame["age"].where(frame["age"].between(18, 100))
    )
    .reset_index(drop=True)
)
assert eda_table["patient_id"].is_unique
print({"analysis_shape": eda_table.shape})


### 3-2. 단변량 분포와 데이터 품질

#### 3-2-1. target과 categorical 분포

binary target은 class count와 rate를 함께 확인합니다. category는 frequency와 target rate를 분리해서 봅니다. category별 target rate 차이는 표본 구성에 대한 관찰이며, 원인이나 feature 유효성을 뜻하지 않습니다.

In [ ]:
target_summary = eda_table["target"].value_counts().sort_index().to_frame("count")
target_summary["rate"] = target_summary["count"] / len(eda_table)
category_summary = eda_table.groupby("icu_type", observed=True).agg(
    patient_count=("patient_id", "size"),
    target_rate=("target", "mean"),
).sort_values("patient_count", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), layout="constrained")
target_summary["count"].plot.bar(ax=axes[0], color=BLUE, edgecolor=INK)
axes[0].set(title="Target class count", xlabel="Target", ylabel="Patients")
category_summary["target_rate"].plot.bar(ax=axes[1], color=ORANGE, edgecolor=INK)
axes[1].set(title="Target rate by ICU type", xlabel="ICU type", ylabel="Target rate")
for axis in axes:
    axis.grid(axis="y", color=GRID, linewidth=0.8)
    axis.set_axisbelow(True)
plt.show()
print(target_summary)
category_summary


#### 3-2-2. numeric 분포와 outlier 후보

`describe()`와 quantile은 중심, spread와 tail을 요약합니다. Mean은 모든 값의 크기에 민감하고 median은 극단값에 덜 민감합니다. Standard deviation은 평균 주위 spread를, IQR은 중간 50%의 범위를 나타냅니다. Histogram은 modality, skew와 tail을, box plot은 group 간 median과 IQR을 빠르게 비교하지만 세부 분포 형태를 숨길 수 있습니다. 통계적으로 드문 값이 곧 오류는 아니므로 source range와 단위를 확인하기 전에 임의로 제거하지 않습니다.

In [ ]:
numeric_columns = [
    "age",
    "heart_rate__mean",
    "lactate__max",
    "urine__sum",
    "noise_feature",
]
numeric_summary = eda_table[numeric_columns].describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
).T

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), layout="constrained")
eda_table["lactate__max"].plot.hist(
    bins=18, ax=axes[0], color=BLUE, edgecolor="white"
)
axes[0].set(title="Lactate maximum distribution", xlabel="Synthetic value", ylabel="Patients")
eda_table.boxplot(column="heart_rate__mean", by="target", ax=axes[1], grid=False)
axes[1].set(title="Heart-rate mean by target", xlabel="Target", ylabel="Synthetic value")
fig.suptitle("")
for axis in axes:
    axis.grid(axis="y", color=GRID, linewidth=0.8)
plt.show()
numeric_summary


#### 3-2-3. 결측률과 missingness-target 관계

결측률은 column별로 정렬해 확인합니다. Missingness는 관측값과 무관한 MCAR, 관측된 정보로 설명되는 MAR, 관측되지 않은 값 자체와 관련된 MNAR로 개념적으로 구분할 수 있지만 EDA만으로 mechanism을 확정하기는 어렵습니다. 결측 자체가 target과 관계를 보일 수 있는 이유도 측정 과정이나 care process의 차이일 수 있습니다. Missing indicator를 feature로 사용할지는 prediction 시점의 가용성과 운영 환경에서 같은 측정 패턴이 유지되는지까지 검토해야 합니다.

In [ ]:
missing_profile = pd.DataFrame(
    {
        "missing_count": eda_table.isna().sum(),
        "missing_rate": eda_table.isna().mean(),
    }
).sort_values("missing_rate", ascending=False)
lactate_missing_target = eda_table.assign(
    lactate_missing=eda_table["lactate__max"].isna()
).groupby("lactate_missing")["target"].agg(["size", "mean"])

nonzero_missing = missing_profile.query("missing_count > 0")
axis = nonzero_missing["missing_rate"].sort_values().plot.barh(
    figsize=(7, 2.6), color=GOLD, edgecolor=INK
)
axis.set(title="Missing rate by feature", xlabel="Missing rate", ylabel="Feature")
axis.grid(axis="x", color=GRID, linewidth=0.8)
plt.show()
print(lactate_missing_target)
missing_profile.head()


### 3-3. Feature 관계와 target association

#### 3-3-1. Pearson과 Spearman correlation

Pearson correlation은 선형 관계를, Spearman correlation은 rank의 단조 관계를 요약합니다. Binary target과 numeric feature의 Pearson 값은 point-biserial association과 같은 형태로 해석할 수 있습니다. 두 방법 모두 관계의 강도와 방향을 하나의 수로 압축하므로 subgroup, nonlinear pattern과 interaction을 놓칠 수 있습니다. Pearson은 outlier에 특히 민감하고, 결측은 pairwise로 제외되어 feature 쌍마다 sample이 달라질 수 있습니다. 여러 feature를 동시에 탐색하면 우연히 큰 값도 나타납니다. Correlation은 원인도, 독립적인 validation 성능도 아닙니다.

In [ ]:
association_columns = [
    "age",
    "heart_rate__mean",
    "heart_rate__mean_copy",
    "lactate__max",
    "urine__sum",
    "noise_feature",
    "future_outcome_score",
]
pearson_with_target = eda_table[association_columns].corrwith(
    eda_table["target"], method="pearson"
)
spearman_with_target = eda_table[association_columns].corrwith(
    eda_table["target"], method="spearman"
)
target_association = pd.DataFrame(
    {"pearson": pearson_with_target, "spearman": spearman_with_target}
).sort_values("pearson", key=lambda values: values.abs(), ascending=False)
target_association


#### 3-3-2. Feature-feature correlation과 redundancy

feature-feature correlation matrix는 거의 같은 정보를 담는 후보를 찾는 데 유용합니다. 높은 correlation만으로 하나를 즉시 제거하지 말고 정의, 측정 안정성, 결측률과 prediction 시점의 가용성을 비교합니다. correlated feature가 여러 개면 model coefficient와 importance가 나뉘거나 불안정해질 수 있습니다.

In [ ]:
feature_correlation = eda_table[association_columns[:-1]].corr(method="spearman")
fig, axis = plt.subplots(figsize=(7.2, 5.8), layout="constrained")
image = axis.imshow(feature_correlation, vmin=-1, vmax=1, cmap="RdBu_r")
axis.set(
    title="Spearman correlation among candidate features",
    xticks=range(len(feature_correlation.columns)),
    yticks=range(len(feature_correlation.index)),
    xticklabels=feature_correlation.columns,
    yticklabels=feature_correlation.index,
)
axis.tick_params(axis="x", rotation=60)
fig.colorbar(image, ax=axis, label="Spearman correlation")
plt.show()
redundant_correlation = feature_correlation.loc[
    "heart_rate__mean", "heart_rate__mean_copy"
]
print({"heart_rate_copy_correlation": redundant_correlation})


### 3-4. Split 안정성과 EDA 기록

#### 3-4-1. Train과 validation profile 비교

feature 판단은 train과 validation 범위에서 수행합니다. stratified split은 target 비율을 비슷하게 유지하지만 feature 분포와 missing rate까지 같게 보장하지 않습니다. split별 profile 차이가 크면 sampling, cohort 또는 시간 drift 가능성을 조사합니다.

In [ ]:
train_table, valid_table = train_test_split(
    eda_table,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=eda_table["target"],
)
split_profile = pd.DataFrame(
    {
        "train": [
            len(train_table),
            train_table["target"].mean(),
            train_table["lactate__max"].isna().mean(),
            train_table["heart_rate__mean"].mean(),
        ],
        "valid": [
            len(valid_table),
            valid_table["target"].mean(),
            valid_table["lactate__max"].isna().mean(),
            valid_table["heart_rate__mean"].mean(),
        ],
    },
    index=["row_count", "target_rate", "lactate_missing_rate", "heart_rate_mean"],
)
split_profile["absolute_gap"] = (split_profile["train"] - split_profile["valid"]).abs()
split_profile


#### 3-4-2. 관찰, 위험과 다음 검증 분리

EDA finding은 관찰한 evidence, 가능한 위험, 다음 검증을 구분해 기록합니다. 아래 표는 합성 데이터에서 의도한 사례를 정리합니다. `future_outcome_score`처럼 강한 association을 보이더라도 prediction 시점에 사용할 수 없으면 성능 비교 전에 제외합니다.

In [ ]:
eda_findings = pd.DataFrame(
    [
        {
            "observation": "future_outcome_score has near-perfect target association",
            "risk": "post-prediction leakage",
            "next_check": "exclude using feature provenance before modeling",
        },
        {
            "observation": "heart-rate mean and copy are strongly correlated",
            "risk": "redundancy and unstable importance",
            "next_check": "compare definitions and keep one representative",
        },
        {
            "observation": "lactate has missing values",
            "risk": "measurement-process dependence",
            "next_check": "evaluate missing indicator and split stability",
        },
    ]
)
eda_findings


### 3-5. EDA 결과를 해석하는 경계

#### 3-5-1. 기술, 추론, 예측과 인과 구분

기술 통계는 현재 sample을 요약하고, 통계적 추론은 sampling uncertainty를 고려해 population에 대한 주장을 평가합니다. 예측 분석은 새로운 sample에서 target을 얼마나 잘 맞추는지 검증하며, 인과 분석은 intervention이 outcome을 바꾸는지 별도의 가정과 설계로 묻습니다. EDA chart와 correlation만으로 네 질문을 서로 대신할 수 없습니다.

또한 같은 data에서 많은 관계를 탐색하고 가장 큰 값만 보고하면 selection bias가 생깁니다. EDA에서 발견한 pattern은 가설로 기록하고, feature 판단은 train/CV와 별도 validation에서 재확인합니다.

In [ ]:
eda_method_guide = pd.DataFrame(
    [
        {
            "question": "현재 sample의 분포와 결함은 무엇인가?",
            "evidence": "count, quantile, histogram, missing profile",
            "cannot_conclude": "population 일반화와 원인",
        },
        {
            "question": "두 변수가 함께 변하는가?",
            "evidence": "scatter, grouped summary, correlation",
            "cannot_conclude": "인과관계와 독립적인 predictive value",
        },
        {
            "question": "새 sample에서도 target을 예측하는가?",
            "evidence": "train/CV and held-out validation metric",
            "cannot_conclude": "intervention effect",
        },
        {
            "question": "변수를 바꾸면 outcome이 변하는가?",
            "evidence": "causal design and assumptions",
            "cannot_conclude": "EDA association alone",
        },
    ]
)
eda_method_guide


## 4. Checks

EDA가 의도한 grain, 데이터 결함, association과 redundancy를 실제로 찾아냈는지 확인합니다.

In [ ]:
assert raw_candidate_table.shape == (241, 10)
assert duplicate_patient_count == 2
assert invalid_age_count == 1
assert eda_table.shape == (240, 10)
assert eda_table["patient_id"].is_unique
assert missing_profile.loc["lactate__max", "missing_count"] == 28
assert abs(redundant_correlation) > 0.95
assert abs(target_association.loc["future_outcome_score", "pearson"]) > 0.95
assert len(train_table) + len(valid_table) == len(eda_table)
assert len(eda_method_guide) == 4
print("All appendix checks passed.")


## 5. Next Steps

- EDA 시작 전에 prediction 시점, observation window와 row grain을 문장으로 고정합니다.
- schema, key, 범위, 결측과 중복을 정리한 뒤 분포와 관계를 봅니다.
- correlation은 관계를 요약할 뿐 인과관계나 validation 성능을 증명하지 않습니다.
- target과 강하게 연결된 feature일수록 availability와 leakage를 먼저 확인합니다.
- 다음 notebook에서 EDA finding을 candidate 생성, association 비교와 feature contract로 연결합니다.

## 6. References

이 notebook의 설명과 예제는 다음 공식 문서를 기준으로 작성했습니다.

- [pandas.DataFrame.describe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)
- [pandas.DataFrame.corr](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html)
- [pandas.DataFrame.corrwith](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corrwith.html)
- [pandas missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [pandas visualization](https://pandas.pydata.org/docs/user_guide/visualization.html)
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)